# SeaDronesSee ConvNeXt QAT — Kaggle
Mỗi lần chạy đúng **1 epoch**, validation + benchmark 100 ảnh, lưu checkpoint vào `/kaggle/working`, sau đó tạo phiên bản mới của Kaggle checkpoint dataset để phiên sau resume. Bật **Settings → Accelerator → GPU** và **Internet** trước khi chạy.

In [1]:
import os, shutil, subprocess, sys, datetime, json
from pathlib import Path
import torch
HAS_CUDA = torch.cuda.is_available()
print('CUDA available:', HAS_CUDA)
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
else:
    print('CPU mode: phù hợp để bootstrap/upload dataset; bật GPU trước khi train')

CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56219482421875


## 1. Clone và cài project
Internet phải được bật để clone GitHub.

In [2]:
REPO = Path('/kaggle/working/EchteAI')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'kagglehub'], check=True)
print('Repository:', Path.cwd())

Cloning into '/kaggle/working/EchteAI'...


Repository: /kaggle/working/EchteAI


## 2. Khai báo Kaggle Dataset
Bạn chỉ cần nhập username Kaggle. Notebook tự tạo **Private Dataset** `seadronessee-compressed` nếu chưa tồn tại, tải dữ liệu từ Tübingen và upload lần đầu. Dataset checkpoint cũng được tự tạo private sau epoch đầu. Những phiên sau tự tải version mới nhất.

In [3]:
import kagglehub
KAGGLE_USERNAME = 'nguyenducthangtb'
SEADRONESSEE_DATASET = f'{KAGGLE_USERNAME}/seadronessee-compressed'
CHECKPOINT_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-checkpoints'
STAGE = 'fp32'  # fp32 hoặc qat
VARIANT = 'M3'
assert KAGGLE_USERNAME != 'YOUR_USERNAME', 'Hãy nhập username Kaggle của bạn'
print('Data:', SEADRONESSEE_DATASET)
print('Checkpoints:', CHECKPOINT_DATASET)
print('Next stage:', STAGE)

Data: nguyenducthangtb/seadronessee-compressed
Checkpoints: nguyenducthangtb/echteai-seadronessee-m3-checkpoints
Next stage: fp32


In [4]:
def find_dataset_root(start):
    start = Path(start)
    candidates = [start, *[p.parent.parent for p in start.rglob('instances_train.json')]]
    for candidate in candidates:
        if (candidate/'annotations/instances_train.json').exists() and (candidate/'images/train').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy cấu trúc images/ + annotations/ trong {start}')

DATA_ROOT = None
try:
    DATA_DOWNLOAD = Path(kagglehub.dataset_download(SEADRONESSEE_DATASET))
    DATA_ROOT = find_dataset_root(DATA_DOWNLOAD)
    print('Using SeaDronesSee already stored on Kaggle')
except Exception as error:
    print('SeaDronesSee private dataset is missing or incomplete:', error)
    print('Bootstrapping it automatically from Tübingen...')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'rclone'], check=True)
    DAV_URL = 'https://cloud.cs.uni-tuebingen.de/public.php/dav/files/ZZxX65FGnQ8zjBP/'
    subprocess.run(['rclone', 'config', 'delete', 'seadrones'], check=False, capture_output=True)
    subprocess.run(['rclone', 'config', 'create', 'seadrones', 'webdav', 'url', DAV_URL, 'vendor', 'nextcloud'], check=True)
    DATA_ROOT = Path('/kaggle/working/seadronessee')
    subprocess.run([
        'rclone', 'copy', 'seadrones:Compressed Version', str(DATA_ROOT),
        '--progress', '--transfers', '2', '--checkers', '4', '--retries', '10',
    ], check=True)
    DATA_ROOT = find_dataset_root(DATA_ROOT)
    print('Uploading SeaDronesSee to the private Kaggle dataset...')
    kagglehub.dataset_upload(
        SEADRONESSEE_DATASET, str(DATA_ROOT),
        version_notes='SeaDronesSee compressed images and COCO annotations',
    )
    print('SeaDronesSee Kaggle dataset version uploaded')
print('Dataset root:', DATA_ROOT)
print('Train images:', len(list((DATA_ROOT/'images/train').glob('*'))))

SeaDronesSee private dataset is missing or incomplete: POST failed with: {"errors":["New Datasets cannot be attached in non-interactive sessions. Found no versions attached for Dataset [nguyenducthangtb/seadronessee-compressed]."],"error":{"code":9},"wasSuccessful":false}
Bootstrapping it automatically from Tübingen...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package rclone.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../rclone_1.53.3-4ubuntu1.22.04.4_amd64.deb ...
Unpacking rclone (1.53.3-4ubuntu1.22.04.4) ...
Setting up rclone (1.53.3-4ubuntu1.22.04.4) ...
Processing triggers for man-db (2.10.2-1) ...
Remote config
--------------------
[seadrones]
url = https://cloud.cs.uni-tuebingen.de/public.php/dav/files/ZZxX65FGnQ8zjBP/
vendor = nextcloud
--------------------
Transferred:   	         0 / 0 Bytes, -, 0 Bytes/s, ETA -
Transferred:            0 / 2, 0%
Elapsed time:         0.9s
Transferring:
 *              annotations/instances_train.json: transferring
 *                annotations/instances_val.json: transferringTransferred:   	    1.992M / 11.204 MBytes, 18%, 3.442 MBytes/s, ETA 2s
Transferred:            0 / 2, 0%
Elapsed time:         1.4s
Transferring:
 *              annotations/instances_train.json: 10% /9.572M, 0/s, -
 *                annotations

Uploading: 100%|██████████| 9.96G/9.96G [01:49<00:00, 91.1MB/s]

Upload successful: /tmp/tmp1h07ck_e/archive.zip (9GB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/nguyenducthangtb/seadronessee-compressed
SeaDronesSee Kaggle dataset version uploaded
Dataset root: /kaggle/working/seadronessee
Train images: 8930


## 3. Khôi phục checkpoint của phiên trước
Kaggle input là read-only, vì vậy checkpoint được copy sang `/kaggle/working/checkpoints`.

In [5]:
OUTPUT = Path('/kaggle/working/checkpoints')
OUTPUT.mkdir(parents=True, exist_ok=True)
try:
    previous = Path(kagglehub.dataset_download(CHECKPOINT_DATASET))
    copied = 0
    for source in previous.rglob('*'):
        if source.is_file():
            shutil.copy2(source, OUTPUT/source.name)
            copied += 1
    print(f'Restored {copied} checkpoint/output files from {previous}')
except Exception as error:
    print('No previous checkpoint dataset; starting a new run:', error)

No previous checkpoint dataset; starting a new run: POST failed with: {"errors":["New Datasets cannot be attached in non-interactive sessions. Found no versions attached for Dataset [nguyenducthangtb/echteai-seadronessee-m3-checkpoints]."],"error":{"code":9},"wasSuccessful":false}


## 4. Tạo config runtime cho Kaggle

In [6]:
import yaml
base = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(DATA_ROOT/'images/train'),
    'train_annotations': str(DATA_ROOT/'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT/'images/val'),
    'val_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT/'images/val'),
    'test_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
})
base['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(OUTPUT/'fp32_best.pt'), 'fp32_last': str(OUTPUT/'fp32_last.pt'),
    'qat_best': str(OUTPUT/'qat_best.pt'), 'qat_last': str(OUTPUT/'qat_last.pt'),
    'int8_model': str(OUTPUT/'selective_int8.pt'),
    'evaluation_json': str(OUTPUT/'evaluation.json'),
    'benchmark_json': str(OUTPUT/'benchmark.json'),
    'epoch_benchmarks': str(OUTPUT/'epoch_benchmarks.json'),
}
RUNTIME_CONFIG = Path('/kaggle/working/seadronessee_kaggle.yaml')
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False))
print(RUNTIME_CONFIG.read_text())

seed: 42
device: auto
dataset:
  train_images: /kaggle/working/seadronessee/images/train
  train_annotations: /kaggle/working/seadronessee/annotations/instances_train.json
  val_images: /kaggle/working/seadronessee/images/val
  val_annotations: /kaggle/working/seadronessee/annotations/instances_val.json
  test_images: /kaggle/working/seadronessee/images/val
  test_annotations: /kaggle/working/seadronessee/annotations/instances_val.json
  ignore_category_ids:
  - 0
  num_classes: 6
  workers: 2
augmentation:
  horizontal_flip_probability: 0.5
  brightness: 0.1
  contrast: 0.1
  saturation: 0.05
  hue: 0.02
model:
  backbone: convnext_tiny
  pretrained_backbone: true
  trainable_backbone_layers: 4
  fpn_out_channels: 256
  min_size: 960
  train_min_sizes:
  - 800
  - 896
  - 960
  max_size: 1600
  anchor_sizes:
  - 8
  - 16
  - 32
  - 64
  - 128
  aspect_ratios:
  - 0.5
  - 1.0
  - 2.0
  rpn_pre_nms_top_n_train: 2000
  rpn_pre_nms_top_n_test: 1000
  rpn_post_nms_top_n_train: 1000
  rpn_p

## 5. Train đúng epoch tiếp theo
FP32 phải hoàn tất 30/30 trước khi đổi `STAGE='qat'`. Mỗi lần chạy cell này chỉ chạy một epoch rồi thoát.

In [7]:
def run_and_log(command, log_path):
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(command), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in process.stdout:
            print(line, end='', flush=True); log_file.write(line); log_file.flush()
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

assert torch.cuda.is_available(), 'Dataset đã sẵn sàng; hãy bật GPU trong Kaggle Settings trước khi train'
command = [sys.executable, '-u', 'scripts/train_next_epoch.py', '--config', str(RUNTIME_CONFIG), '--stage', STAGE]
if STAGE == 'qat': command += ['--variant', VARIANT]
run_and_log(command, OUTPUT/f'{STAGE}_train.log')

Command: /usr/bin/python3 -u scripts/train_next_epoch.py --config /kaggle/working/seadronessee_kaggle.yaml --stage fp32
Starting FP32 epoch 1/30
Checkpoint: /kaggle/working/checkpoints/fp32_last.pt
Command: /usr/bin/python3 -u scripts/train_fp32.py --config /kaggle/working/seadronessee_kaggle.yaml --epochs-this-run 1
FP32 setup device=cuda train_images=8930 val_images=1547 epochs=30
FP32 best=/kaggle/working/checkpoints/fp32_best.pt
FP32 last=/kaggle/working/checkpoints/fp32_last.pt
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth

  0%|          | 0.00/109M [00:00<?, ?B/s]
 18%|█▊        | 19.5M/109M [00:00<00:00, 203MB/s]
 37%|███▋      | 40.6M/109M [00:00<00:00, 214MB/s]
 57%|█████▋    | 61.9M/109M [00:00<00:00, 218MB/s]
 76%|███████▋  | 83.2M/109M [00:00<00:00, 220MB/s]
 96%|█████████▌| 104M/109M [00:00<00:00, 220MB/s] 
100%|██████████| 109M/109M [00:00<00:00, 219MB/s]
model=convnext_tiny 

## 6. Kiểm tra rồi lưu checkpoint thành Kaggle Dataset version
Không đóng phiên trước khi cell upload hoàn tất. Nếu checkpoint dataset chưa tồn tại, KaggleHub tự tạo dataset private; nếu đã có, nó tạo version mới.

In [8]:
def checkpoint_epoch(path):
    path = Path(path)
    return int(torch.load(path, map_location='cpu', weights_only=False).get('epoch', 0)) if path.exists() else 0

fp32_epoch = checkpoint_epoch(OUTPUT/'fp32_last.pt')
qat_epoch = checkpoint_epoch(OUTPUT/'qat_last.pt')
print(f'FP32={fp32_epoch}/30 QAT={qat_epoch}/11')
for path in sorted(OUTPUT.glob('*')):
    print(f'{path.name:30s} {path.stat().st_size/2**20:9.2f} MB')
kagglehub.dataset_upload(
    CHECKPOINT_DATASET, str(OUTPUT),
    version_notes=f'FP32 epoch {fp32_epoch}, QAT epoch {qat_epoch}',
)
print('Checkpoint dataset uploaded:', CHECKPOINT_DATASET)

FP32=1/30 QAT=0/11
epoch_benchmarks.json               0.00 MB
fp32_best.pt                      516.02 MB
fp32_last.pt                      516.02 MB
fp32_train.log                      0.01 MB
Uploading Dataset https://api.kaggle.com/datasets/nguyenducthangtb/echteai-seadronessee-m3-checkpoints ...
Starting upload for file /kaggle/working/checkpoints/epoch_benchmarks.json


Uploading: 100%|██████████| 202/202 [00:00<00:00, 734B/s]

Upload successful: /kaggle/working/checkpoints/epoch_benchmarks.json (202B)
Starting upload for file /kaggle/working/checkpoints/fp32_last.pt



Uploading: 100%|██████████| 541M/541M [00:07<00:00, 76.4MB/s]

Upload successful: /kaggle/working/checkpoints/fp32_last.pt (516MB)
Starting upload for file /kaggle/working/checkpoints/fp32_train.log



Uploading: 100%|██████████| 8.79k/8.79k [00:00<00:00, 31.7kB/s]

Upload successful: /kaggle/working/checkpoints/fp32_train.log (9KB)
Starting upload for file /kaggle/working/checkpoints/fp32_best.pt



Uploading: 100%|██████████| 541M/541M [00:05<00:00, 91.2MB/s]

Upload successful: /kaggle/working/checkpoints/fp32_best.pt (516MB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/nguyenducthangtb/echteai-seadronessee-m3-checkpoints
Checkpoint dataset uploaded: nguyenducthangtb/echteai-seadronessee-m3-checkpoints
